# ParamISP demo - Google Colab

Official demo for **ParamISP: Learned Forward and Inverse ISPs using Camera Parameters** (CVPR 2024).

Two directions: **sRGB -> RAW** (inverse ISP) and **RAW -> sRGB** (forward ISP), controlled by camera parameters (ISO, exposure, focal length, f-number, white balance, colour matrix).

**No GPU is required** - the model is only ~0.7M parameters and runs on CPU.
To use a free GPU instead: **Runtime > Change runtime type > T4 GPU**.

Running the last cell prints a public `*.gradio.live` link (valid ~72h).

In [ ]:
!git clone https://github.com/gliderest/paramISP.git
%cd paramISP

In [ ]:
# Runtime dependencies (torch is already provided by Colab).
!pip install -q "numpy<2" scipy pillow tifffile exifread matplotlib toolz pyyaml \
  python-dotenv rawpy gdown "kornia>=0.7,<0.9" "gradio==4.44.1" "gradio_client==1.3.0" \
  "pydantic==2.9.2" "fastapi==0.115.0" "starlette==0.38.6" "uvicorn==0.30.6" \
  "jinja2<3.2" "huggingface_hub==0.24.7" spaces

In [ ]:
# Quick check: the network builds and runs (weights download on first use).
import os
os.environ["PYTHONPATH"] = "."
from paramisp_demo import dataio, inference, weights
import numpy as np

device = inference.resolve_device()
print("device:", device)
model = inference.load_model(weights.ensure_weights("inverse", "D7000"), True, device)[0]
raw = inference.predict_srgb_to_raw(
    model, dataio.load_srgb("assets/example_srgb.jpg"), device,
    camera_model="D7000", bayer_pattern="RGGB", white_balance=(2.0, 1.0, 1.5),
)
print("predicted RAW:", raw.shape, float(raw.min()), float(raw.max()))

In [ ]:
# Launch the interactive Gradio demo with a free public link.
from app import build_demo
build_demo().queue().launch(share=True)